In [ ]:
import pandas as pd

In [ ]:
expense_dataset = pd.read_csv('./data/Personal_Finance_Dataset.csv')

expense_dataset = expense_dataset.sort_values("Date")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(expense_dataset ['Date'], expense_dataset['Amount'], marker='o')
plt.xlabel("Date")
plt.ylabel("Amount")
plt.title("Amount Over Time")
plt.grid(True)
plt.tight_layout()

In [ ]:
plt.hist(expense_dataset['Amount'],bins=50)

In [ ]:
plt.boxplot(expense_dataset['Amount'])

# Cullen-Freychart

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
from sklearn.utils import resample

np.random.seed(42)
# Simulating a dataset that looks like expense data (e.g., Lognormal)
data_size = 5000 
expense_data = np.random.lognormal(mean=5.5, sigma=1.2, size=data_size)

# Add a few very small expenses
expense_data = np.concatenate([expense_data, np.random.uniform(10, 50, 500)])
sample_expense_dataset = pd.DataFrame({'Amount': expense_data})
# --------------------------------------------------------------------------

# Set the data series
amount_data = expense_dataset['Amount']

In [ ]:
def run_bootstrapped_cullen_frey(data_series, n_bootstraps=1000):
    """
    Performs bootstrap sampling and calculates the squared skewness and 
    kurtosis for each sample.
    """
    skew_sq_values = []
    kurtosis_values = []
    
    # Get the size of the original dataset
    n_samples = len(data_series)
    
    for i in range(n_bootstraps):
        # 1. Create a bootstrap sample (resampling with replacement)
        sample = resample(data_series.values, replace=True, n_samples=n_samples)
        
        # 2. Calculate the moments for the bootstrap sample
        sample_skewness = skew(sample)
        sample_kurtosis = kurtosis(sample, fisher=False) # Pearson's Kurtosis (Normal=3)
        
        # 3. Store the results
        skew_sq_values.append(sample_skewness**2)
        kurtosis_values.append(sample_kurtosis)
        
    # Return a DataFrame of the results
    return pd.DataFrame({
        'Skewness_Squared': skew_sq_values,
        'Kurtosis': kurtosis_values
    })

In [ ]:
cullen_frey_boot_df = run_bootstrapped_cullen_frey(expense_dataset['Amount'])

In [ ]:
## Calculate Skewness and Kurtosis

# Calculate Sample Skewness (g1)
def skew_kertosis(amount_data,cullen_frey_boot_df):
    sample_skewness = skew(amount_data)

    # Calculate Sample Kurtosis (g2 + 3)
    # We use fisher=False to get the Pearson/moment kurtosis (where Normal = 3)
    sample_kurtosis = kurtosis(amount_data, fisher=False)

    # Calculate the squared skewness for the plot's X-axis
    skewness_squared = sample_skewness**2

    max_x = cullen_frey_boot_df['Skewness_Squared'].max()
    max_y = cullen_frey_boot_df['Kurtosis'].max()
    print(f"Sample Skewness^2: {skewness_squared:.4f}")
    print(f"Sample Kurtosis: {sample_kurtosis:.4f}")


    ## Create the Cullen-Frey Plot

    plt.figure(figsize=(9, 6))

    # --- 1. Plot the Sample Data Point (Your 'Amount' data) ---
    plt.scatter(cullen_frey_boot_df['Skewness_Squared'], 
                cullen_frey_boot_df['Kurtosis'], 
                color='blue', s=10, alpha=0.2, label=f'Bootstrap Samples (n={len(cullen_frey_boot_df)})')
    plt.scatter(skewness_squared, sample_kurtosis, 
                color='blue', s=150, marker='o', 
                label='Your Sample Data (Amount)')

    # --- 2. Plot Theoretical Distribution Points ---
    # These are the expected (Skew^2, Kurtosis) for common distributions
    # Normal Distribution: (0, 3)
    plt.scatter(0, 3, color='red', s=150, marker='X', label='Normal (0, 3)')

    # Uniform Distribution: (0, 1.8)
    plt.scatter(0, 1.8, color='green', s=150, marker='D', label='Uniform (0, 1.8)')

    # Exponential Distribution: (4, 9)
    plt.scatter(4, 9, color='orange', s=150, marker='s', label='Exponential (4, 9)')

    # --- 3. Add Theoretical Distribution Curves (Optional but Helpful) ---
    # Gamma and Beta curves are often plotted to help identify candidates.
    # They show the space where these distributions can exist.

    # Gamma Curve (Kurtosis = 3/2 * Skewness^2 + 3)
    x_gamma = np.linspace(0, 10, 100)
    y_gamma = 1.5 * x_gamma + 3
    plt.plot(x_gamma, y_gamma, color='gray', linestyle='--', label='Gamma Family')

    # Lognormal Family: Falls just above the Gamma line (not easy to plot exactly)
    # Weibull Family: Tends to fall between the Gamma and Uniform regions
    plt.scatter(0, 4.2, color='purple', s=150, marker='^', label='Logistic (0, 4.2)')

    ### Lognormal Family
    # The Lognormal curve is complex, but it starts at the Normal point (0, 3) and generally 
    # lies *just above* the Gamma line, especially for high skewness.
    # Plotting the Lognormal curve's exact line requires solving for the shape parameter (m), 
    # but it's often shown as a distinct, slightly curved line above Gamma.
    # We'll plot an approximation line to illustrate the separation.
    y_lognorm_approx = 1.7 * x_gamma + 3.1 
    plt.plot(x_gamma, y_lognorm_approx, color='darkgreen', linestyle=':', linewidth=2, label='Lognormal Family')


    ### Beta Family
    # The Beta distribution occupies the entire region *below* the Gamma line and *above* # the region defined by the Uniform point (0, 1.8) and the curve connecting it to the Normal point.
    # Beta is the most flexible and covers the entire region below the Gamma line.
    # We often highlight the region rather than a specific line.
    # (Note: Shading the region is best, but using a single line often marks the boundary.)

    # --- 4. Customize the Plot ---
    plt.title('Cullen-Frey Graph for Expense Amount Distribution', fontsize=14)
    plt.xlabel('Squared Skewness $(\\text{Skew}^2)$', fontsize=12)
    plt.ylabel('Kurtosis', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.ylim(0, max(sample_kurtosis + 2, 10)) # Adjust Y-limit dynamically
    plt.xlim(0, max(skewness_squared + 1, 5)) # Adjust X-limit dynamically
    plt.show()

In [ ]:
skew_kertosis(amount_data,cullen_frey_boot_df)

# Candiates: Beta,gamma

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gamma, beta


# --- 2. Fit Distributions using MLE ---

# A. Fit Gamma Distribution (3 parameters: a, loc, scale)
# Fix location (minimum value) at 0 as data is non-negative
params_gamma = gamma.fit(amount_data, floc=0)
a_gamma, loc_gamma, scale_gamma = params_gamma
print(f"Gamma Fit Parameters: a={a_gamma:.2f}, loc={loc_gamma:.2f}, scale={scale_gamma:.2f}")

# B. Fit Beta Distribution (4 parameters: a, b, loc, scale)
# Let fit estimate the bounds (loc and scale)
params_beta = beta.fit(amount_data, floc=0)
a_beta, b_beta, loc_beta, scale_beta = params_beta
print(f"Beta Fit Parameters: a={a_beta:.2f}, b={b_beta:.2f}, loc={loc_beta:.2f}, scale={scale_beta:.2f}")

# --- 3. Prepare Data for Plotting ---
data_min = amount_data.min()
data_max = amount_data.max()
# Create a range of x-values for plotting the theoretical curves
x = np.linspace(data_min, data_max * 1.05, 1000)

# Calculate theoretical PDF and CDF values
pdf_gamma = gamma.pdf(x, a_gamma, loc=loc_gamma, scale=scale_gamma)
cdf_gamma = gamma.cdf(x, a_gamma, loc=loc_gamma, scale=scale_gamma)

pdf_beta = beta.pdf(x, a_beta, b_beta, loc=loc_beta, scale=scale_beta)
cdf_beta = beta.cdf(x, a_beta, b_beta, loc=loc_beta, scale=scale_beta)

# --- 4. Plotting (PDF and CDF) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left Plot: Density / PDF ---
axes[0].hist(amount_data, bins=50, density=True, alpha=0.6, color='skyblue', label='Actual Data (Histogram)')
axes[0].plot(x, pdf_gamma, 'r-', lw=2, label='Fitted Gamma PDF')
axes[0].plot(x, pdf_beta, 'g--', lw=2, label='Fitted Beta PDF')

axes[0].set_title('Density (PDF) Comparison: Data vs. Fitted Distributions', fontsize=14)
axes[0].set_xlabel('Amount', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].legend(loc='upper right')
# Zoom in to the main data range (up to 99th percentile) for clarity
axes[0].set_xlim(amount_data.min() * 0.9, amount_data.quantile(0.99) * 1.1)
axes[0].grid(True, linestyle=':', alpha=0.5)

# --- Right Plot: CDF ---
# Calculate Empirical CDF (ECDF) for the actual data
ecdf_y = np.arange(1, len(amount_data) + 1) / len(amount_data)
ecdf_x = np.sort(amount_data)

axes[1].plot(ecdf_x, ecdf_y, 'b.', markersize=5, alpha=0.4, label='Actual Data (ECDF)')
axes[1].plot(x, cdf_gamma, 'r-', lw=2, label='Fitted Gamma CDF')
axes[1].plot(x, cdf_beta, 'g--', lw=2, label='Fitted Beta CDF')

axes[1].set_title('Cumulative Distribution Function (CDF) Comparison', fontsize=14)
axes[1].set_xlabel('Amount', fontsize=12)
axes[1].set_ylabel('Cumulative Probability', fontsize=12)
axes[1].legend(loc='lower right')
# Zoom in to the main data range
axes[1].set_xlim(amount_data.min() * 0.9, amount_data.quantile(0.99) * 1.1)
axes[1].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

# Dataset 2

In [ ]:
expense_dataset_2 = pd.read_csv('./data/expenses_income_summary.csv')

expense_dataset_2 = expense_dataset_2.sort_values("Date")

In [ ]:
import numpy as np

In [ ]:
plt.hist(expense_dataset_2['amount'],bins=5)
plt.xticks(np.arange(0, 5000, 1000))

In [ ]:
expense_dataset_2['amount'] = pd.to_numeric(
    expense_dataset_2['amount'], 
    errors='coerce'
)

# 2. (Optional but recommended) Remove NaN values
# Boxplot can usually handle NaNs, but it's good practice to clean the data
expense_data_clean = expense_dataset_2.dropna(subset=['amount'])

# 3. Create the box plot with the cleaned, numeric data
plt.boxplot(expense_data_clean['amount'])
plt.title("Box Plot of Expense Amount")
plt.ylabel("Amount")
plt.show()

In [ ]:
expense_data_clean = expense_dataset_2.dropna(subset=['amount'])
cullen_frey_boot_df = run_bootstrapped_cullen_frey(expense_data_clean['amount'])
amount_data = expense_data_clean['amount']
skew_kertosis(amount_data,cullen_frey_boot_df)

In [ ]:
ad = amount_data[amount_data > 0]
amount_data = ad

# Candidates are Beta, exonential and gamma

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gamma, beta


# --- 2. Fit Distributions using MLE ---

# A. Fit Gamma Distribution (3 parameters: a, loc, scale)
# Fix location (minimum value) at 0 as data is non-negative
params_gamma = gamma.fit(amount_data, floc=0)
a_gamma, loc_gamma, scale_gamma = params_gamma
print(f"Gamma Fit Parameters: a={a_gamma:.2f}, loc={loc_gamma:.2f}, scale={scale_gamma:.2f}")

# B. Fit Beta Distribution (4 parameters: a, b, loc, scale)
# Let fit estimate the bounds (loc and scale)
params_beta = beta.fit(amount_data, floc=0)
a_beta, b_beta, loc_beta, scale_beta = params_beta
print(f"Beta Fit Parameters: a={a_beta:.2f}, b={b_beta:.2f}, loc={loc_beta:.2f}, scale={scale_beta:.2f}")

# --- 3. Prepare Data for Plotting ---
data_min = amount_data.min()
data_max = amount_data.max()
# Create a range of x-values for plotting the theoretical curves
x = np.linspace(data_min, data_max * 1.05, 1000)

# Calculate theoretical PDF and CDF values
pdf_gamma = gamma.pdf(x, a_gamma, loc=loc_gamma, scale=scale_gamma)
cdf_gamma = gamma.cdf(x, a_gamma, loc=loc_gamma, scale=scale_gamma)

pdf_beta = beta.pdf(x, a_beta, b_beta, loc=loc_beta, scale=scale_beta)
cdf_beta = beta.cdf(x, a_beta, b_beta, loc=loc_beta, scale=scale_beta)

# --- 4. Plotting (PDF and CDF) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left Plot: Density / PDF ---
axes[0].hist(amount_data, bins=50, density=True, alpha=0.6, color='skyblue', label='Actual Data (Histogram)')
axes[0].plot(x, pdf_gamma, 'r-', lw=2, label='Fitted Gamma PDF')
axes[0].plot(x, pdf_beta, 'g--', lw=2, label='Fitted Beta PDF')

axes[0].set_title('Density (PDF) Comparison: Data vs. Fitted Distributions', fontsize=14)
axes[0].set_xlabel('Amount', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].legend(loc='upper right')
# Zoom in to the main data range (up to 99th percentile) for clarity
axes[0].set_xlim(amount_data.min() * 0.9, amount_data.quantile(0.99) * 1.1)
axes[0].grid(True, linestyle=':', alpha=0.5)

# --- Right Plot: CDF ---
# Calculate Empirical CDF (ECDF) for the actual data
ecdf_y = np.arange(1, len(amount_data) + 1) / len(amount_data)
ecdf_x = np.sort(amount_data)

axes[1].plot(ecdf_x, ecdf_y, 'b.', markersize=5, alpha=0.4, label='Actual Data (ECDF)')
axes[1].plot(x, cdf_gamma, 'r-', lw=2, label='Fitted Gamma CDF')
axes[1].plot(x, cdf_beta, 'g--', lw=2, label='Fitted Beta CDF')

axes[1].set_title('Cumulative Distribution Function (CDF) Comparison', fontsize=14)
axes[1].set_xlabel('Amount', fontsize=12)
axes[1].set_ylabel('Cumulative Probability', fontsize=12)
axes[1].legend(loc='lower right')
# Zoom in to the main data range
axes[1].set_xlim(amount_data.min() * 0.9, amount_data.quantile(0.99) * 1.1)
axes[1].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

# Standardizing and normalizing the amounts as both are in beta distributions

In [ ]:
import numpy as np
from scipy.stats import beta
import pandas as pd

expense_data_clean = expense_data_clean[expense_data_clean['amount'] != 0].reset_index(drop=True)

def rescale_to_unit_interval(data):
    # Avoid zero-width range
    min_val, max_val = np.min(data), np.max(data)
    scaled = (data - min_val) / (max_val - min_val)
    # Shift values within (0,1) to avoid exactly 0 or 1 (epsilon=1e-6)
    epsilon = 1e-6
    scaled = np.clip(scaled, epsilon, 1 - epsilon)
    return scaled

def calculate_beta_params_mle(data):
    # The 'fit' method returns (alpha (a), beta (b), location (loc), scale (scale))
    # We force loc=0 and scale=1 for the standard beta distribution
    alpha, beta_val, loc, scale = beta.fit(data, floc=0, fscale=1)
    return alpha, beta_val


data1 = rescale_to_unit_interval(expense_dataset['Amount'].to_numpy())
data2 = rescale_to_unit_interval(expense_data_clean['amount'].to_numpy())

# Calculate parameters for the first dataset
alpha1, beta1 = calculate_beta_params_mle(data1)
print(f"Dataset 1: Alpha = {alpha1:.3f}, Beta = {beta1:.3f}")
alpha2, beta2 = calculate_beta_params_mle(data2)
print(f"Dataset 2: Alpha = {alpha2:.3f}, Beta = {beta2:.3f}")


# # Using the same example data from Method 1
# alpha1_mom, beta1_mom = calculate_beta_params_mom(expense_dataset['Amount'])
# print(f"Dataset 1 (MOM): Alpha = {alpha1_mom:.3f}, Beta = {beta1_mom:.3f}")

# alpha2_mom, beta2_mom = calculate_beta_params_mom(expense_dataset_2['amount'])
# print(f"Dataset 2 (MOM): Alpha = {alpha2_mom:.3f}, Beta = {beta2_mom:.3f}")


# First Approach to check if it gives Results

In [ ]:
expense_data_clean['amount'].min()

In [ ]:
min2 = expense_data_clean['amount'].min()
max2 = expense_data_clean['amount'].max()

In [ ]:
max1 = expense_dataset['Amount'].max()
min1 = expense_dataset['Amount'].min()

In [ ]:
print(max2,min2)
print(max1,min1)

In [ ]:
# Normalize
expense_dataset['Amount_norm'] = (expense_dataset['Amount'] - min1) / (max1 - min1)
expense_data_clean['Amount_norm'] = (expense_data_clean['amount'] - min2) / (max2 - min2)

# Combine normalized data
# Option 1: Stack for a single column of normalized values
combined = pd.concat([expense_dataset[['Amount_norm']], expense_data_clean[['Amount_norm']]], ignore_index=True)

# Option 2: Concatenate full normalized DataFrames (preserving origins)
combined_full = pd.concat([expense_dataset, expense_dataset], ignore_index=True)

print(combined.head())
print(combined_full.head())

In [ ]:
combined_full.head()

In [ ]:
combined_full

# 2. Define the list of entries you want to KEEP
allowed_categories = ['Expense']

# 3. Filter the DataFrame using .isin()
# This creates a boolean mask where True means the row should be kept
filter_mask = combined_full['Type'].isin(allowed_categories)

# 4. Apply the mask to the DataFrame
filtered_df = combined_full[filter_mask]

In [ ]:
filtered_df

In [ ]:
def aggeregate(key,frequecy,filtered_df, aggregate_on_which_param):    
    x = pd.to_datetime(filtered_df)

    # Now the groupby operation should work correctly
    monthly_data = x.groupby(pd.Grouper(key= key, freq= frequecy)).agg(
        {f'{aggregate_on_which_param}': 'sum'}
    ).reset_index()

    print(monthly_data)
    return monthly_data

In [ ]:
monthly_data.to_csv('./data/prepared_data/Aggregated_Dataset_TS.csv')

# Therefore we can see the distribution isnt properly equal so combining them can be disaterous

# Now this will be one of approachs where we combine data 1 and data 2 normalized already with a dataset tage to train XGBoost Model

In [ ]:
expense_dataset['Normalized_Amount'] = data1
expense_dataset.head()

In [ ]:
expense_dataset['dataset'] = 0

In [ ]:
expense_dataset = expense_dataset[['Date', 'Amount', 'Type' ,'Normalized_Amount','dataset']]

In [ ]:
expense_dataset.head()

In [ ]:
expense_dataset_2 = expense_dataset_2[['Date','amount','type']]

In [ ]:
expense_dataset_2.dropna(inplace= True)

In [ ]:
expense_dataset_2['amount'].dropna(inplace= True)
print(expense_dataset_2['amount'].isna().sum())

In [ ]:
expense_dataset_2['amount'].min()

In [ ]:
data1 = rescale_to_unit_interval(expense_dataset['Amount'].to_numpy())
data2 = rescale_to_unit_interval(expense_dataset_2['amount'].to_numpy())

# Calculate parameters for the first dataset
alpha1, beta1 = calculate_beta_params_mle(data1)
print(f"Dataset 1: Alpha = {alpha1:.3f}, Beta = {beta1:.3f}")
alpha2, beta2 = calculate_beta_params_mle(data2)
print(f"Dataset 2: Alpha = {alpha2:.3f}, Beta = {beta2:.3f}")


In [ ]:
expense_dataset_2.drop
expense_dataset_2['Normalized_Amount'] = data2
expense_dataset_2.head()

In [ ]:
expense_dataset_2['dataset'] = 1

In [ ]:
expense_dataset_2.rename(columns={'amount':'Amount', 'type':'Type'}, inplace= True)

In [ ]:
expense_dataset.head()

In [ ]:
expense_dataset_2.head()

In [ ]:
pd.to_datetime(expense_dataset['Date'])

In [ ]:
pd.to_datetime(expense_dataset_2['Date'])

In [ ]:
expense_dataset_2['Date'] = pd.to_datetime(expense_dataset_2['Date'], errors='coerce', dayfirst=False)

In [ ]:
expense_dataset_2['Date'] = expense_dataset_2['Date'].dt.date

In [ ]:
expense_dataset_2.head()

In [ ]:
combined_full = pd.concat([expense_dataset, expense_dataset_2], ignore_index=True)

In [ ]:
combined_full

In [ ]:
combined_full.isna().sum()

In [ ]:
combined_full.to_csv('./data/prepared_data/Forecasting_Master_Dataset.csv')

# We need to use bootstrapping and statistics to combine the Data

# 1) Find mean, median ,mode
# 2) Outliers to be detected
# 3) A way to scale prices according to a parameter(GDP-per capita might work)
# 4) Then do bootstrapping and combine the data (in monthly fasion)
# 5) Combine with the github dataset (similar way)
# 6) Build forecasting model